In [1]:
import h5py
import numpy as np
from pathlib import Path
from tqdm import tqdm
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# --- 1. 核心配置 ---
# 注意：如果你想做Group PCA，必须保证所有人的输入维度一致（即ROI体素数量相同）
# 如果你的h5文件中每个人的体素数量不同，这里必须先做mask intersection或者插值
DATA_DIR = Path("data/ToM_Data/Human_fMRI_Data")
ROI_FILE = "Left_TPJ.h5" 
PERIOD = "Belief_Period_H5" 
SUBJECTS = [f"TOM{i:03d}" for i in range(1, 22) if i != 6]

CONDITIONS = [
    "FB Prot Wrong", "FB Prot Corr", "FB Phys Wrong", "FB Phys Corr",
    "TB Prot Wrong", "TB Prot Corr", "TB Phys Wrong", "TB Phys Corr"
]
print(f"Running: HRF Shift -> Subject Centering -> Global PCA...")

X_list = []      
y_list = []      

for sub_idx, sub in enumerate(tqdm(SUBJECTS)):
    input_file = DATA_DIR / PERIOD / sub / ROI_FILE
    sub_X = []
    sub_y = []
    
    try:
        with h5py.File(input_file, 'r') as f:
            for cond_name in f.keys():
                for trial_name in f[cond_name].keys():
                    group = f[cond_name][trial_name]
                    signal = group['Signal'][()] 
                    label = int(group['Condition_Number'][()])
                    
                    if signal.shape[0] >= 4:
                        valid_signal = signal[-2:, :] 
                        trial_vec = np.mean(valid_signal, axis=0)
                        trial_vec = np.nan_to_num(trial_vec)
                        sub_X.append(trial_vec)
                        sub_y.append(label)
    except Exception as e:
        print(f"Error loading {sub}: {e}")
        continue
    
    if len(sub_X) == 0: continue
        
    sub_X = np.array(sub_X) # Shape: (n_trials, n_voxels)
    sub_y = np.array(sub_y)
    # [关键修正 1]: 移除内部的 SelectKBest
    # 保持原始体素排列，确保列与列之间是解剖对齐的。
    # 假设：所有被试的ROI mask是一样的（MNI空间）。如果不一样，这里会报错或无效。

    # [关键修正 2]: 被试内去中心化 (Subject Centering / De-meaning)
    # 计算该被试所有试次的平均模式（即该人的"平均脑活动"）
    subject_mean_pattern = np.mean(sub_X, axis=0)
    # 减去平均模式 -> 剩下的就是"相对于这个人的基线，任务引起的变化"
    sub_X_centered = sub_X - subject_mean_pattern
    
    # 此时再做标准化，主要是为了统一方差
    # 注意：这里不需要with_mean=True，因为上面已经手动去mean了，不过再做一次也没坏处
    scaler = StandardScaler()
    sub_X_norm = scaler.fit_transform(sub_X_centered)
    
    X_list.append(sub_X_norm)
    y_list.append(sub_y)

# --- 3. 堆叠数据 ---
# 只有当特征在物理意义上对齐（没乱删体素）时，vstack 才有意义
X_final = np.vstack(X_list)
y_final = np.concatenate(y_list)

Running: HRF Shift -> Subject Centering -> Global PCA...


100%|██████████| 20/20 [00:00<00:00, 72.92it/s]


In [ ]:
import h5py
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm

# --- 配置 ---
DATA_DIR = Path("data/ToM_Data/Human_fMRI_Data")
SUBJECTS = [f"TOM{i:03d}" for i in range(1, 22) if i != 6]
CONDITIONS = [
    "FB Prot Wrong", "FB Prot Corr", "FB Phys Wrong", "FB Phys Corr",
    "TB Prot Wrong", "TB Prot Corr", "TB Phys Wrong", "TB Phys Corr"
]

PERIODS = ["Belief_Period_H5", "Perspective_Period_H5"]
ROIS = ["Left_PFC.h5", "Left_TPJ.h5", "Right_PFC.h5", "Right_TPJ.h5"]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT_DIR = Path("data/results_supcon")
OUTPUT_DIR.mkdir(exist_ok=True)

# --- 1. 模型定义 ---
class ContrastiveEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, embed_dim=64):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Linear(256, hidden_dim), 
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU() 
        )
        self.projector = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, embed_dim)
        )

    def forward(self, x):
        feat = self.encoder(x)
        proj = self.projector(feat)
        return feat, F.normalize(proj, dim=1)

class SupConLoss(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        device = features.device
        batch_size = features.shape[0]
        labels = labels.contiguous().view(-1, 1)
        mask = torch.eq(labels, labels.T).float().to(device)

        anchor_dot_contrast = torch.div(torch.matmul(features, features.T), self.temperature)
        logits_max, _ = torch.max(anchor_dot_contrast, dim=1, keepdim=True)
        logits = anchor_dot_contrast - logits_max.detach()

        logits_mask = torch.scatter(
            torch.ones_like(mask), 1, 
            torch.arange(batch_size).view(-1, 1).to(device), 0
        )
        mask = mask * logits_mask

        exp_logits = torch.exp(logits) * logits_mask
        log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True) + 1e-8)

        mean_log_prob_pos = (mask * log_prob).sum(1) / (mask.sum(1) + 1e-8)
        loss = - mean_log_prob_pos.mean()
        return loss

# --- 2. 数据加载函数 (含关键的 Subject Centering) ---
def load_data(period_name, roi_file):
    print(f"\nLoading Data: {period_name} / {roi_file}")
    X_list, y_list = [], []
    
    for sub in SUBJECTS:
        input_path = DATA_DIR / period_name / sub / roi_file
        sub_X, sub_y = [], []
        
        try:
            with h5py.File(input_path, 'r') as f:
                for cond_name in f.keys():
                    for trial_name in f[cond_name].keys():
                        group = f[cond_name][trial_name]
                        signal = group['Signal'][()] 
                        label = int(group['Condition_Number'][()])
                        
                        if signal.shape[0] >= 4:
                            # 取最后2个TR均值
                            valid_signal = signal[-2:, :] 
                            trial_vec = np.mean(valid_signal, axis=0)
                            trial_vec = np.nan_to_num(trial_vec)
                            sub_X.append(trial_vec)
                            sub_y.append(label)
        except Exception as e:
            # print(f"Warning: {sub} missing or error: {e}")
            continue
        
        if len(sub_X) == 0: continue
        sub_X = np.array(sub_X)
        sub_y = np.array(sub_y)
        
        # [核心去噪] Subject Centering
        sub_mean = np.mean(sub_X, axis=0)
        sub_X_centered = sub_X - sub_mean
        
        # 标准化
        scaler = StandardScaler()
        sub_X_norm = scaler.fit_transform(sub_X_centered)
        
        X_list.append(sub_X_norm)
        y_list.append(sub_y)
        
    if not X_list:
        return None, None
        
    return np.vstack(X_list), np.concatenate(y_list)

# --- 3. 训练与提取函数 ---
def train_and_extract(X, y, task_name):
    # Split
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.1, stratify=y, random_state=42)
    
    # Dataset
    train_ds = TensorDataset(torch.FloatTensor(X_train), torch.LongTensor(y_train))
    train_dl = DataLoader(train_ds, batch_size=64, shuffle=True, drop_last=True)
    
    # Init Model
    model = ContrastiveEncoder(input_dim=X.shape[1]).to(DEVICE)
    criterion = SupConLoss(temperature=0.1)
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    
    # Train
    epochs = 50
    loss_history = []
    model.train()
    
    print(f"Training {task_name}...")
    for epoch in range(epochs):
        epoch_loss = 0
        for batch_x, batch_y in train_dl:
            batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)
            optimizer.zero_grad()
            feat, proj = model(batch_x)
            loss = criterion(proj, batch_y)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        loss_history.append(epoch_loss / len(train_dl))
        
    # Extract Embeddings (Full Data)
    model.eval()
    with torch.no_grad():
        full_tensor = torch.FloatTensor(X).to(DEVICE)
        embeddings, _ = model(full_tensor) # 取 feat
        embeddings = embeddings.cpu().numpy()
        
    return embeddings, loss_history

# --- 4. 主流程 ---
final_embeddings = {} # 存储所有结果: key="Period_ROI", value=(embeddings, labels)

for period in PERIODS:
    for roi in ROIS:
        task_id = f"{period}_{roi.replace('.h5', '')}"
        
        # 1. 加载数据
        X_raw, y_raw = load_data(period, roi)
        if X_raw is None:
            print(f"Skipping {task_id} (No Data)")
            continue
            
        # 2. 训练 & 提取
        emb, losses = train_and_extract(X_raw, y_raw, task_id)
        
        # 3. 存储结果
        final_embeddings[task_id] = {
            "embeddings": emb,
            "labels": y_raw,
            "loss": losses
        }
        
        # 4. 绘图并保存 (PCA Check)
        pca = PCA(n_components=2)
        X_pca = pca.fit_transform(emb)
        
        plt.figure(figsize=(10, 6))
        sns.scatterplot(
            x=X_pca[:, 0], y=X_pca[:, 1],
            hue=[CONDITIONS[i] for i in y_raw],
            style=['False Belief' if i < 4 else 'True Belief' for i in y_raw],
            palette='tab10', s=60, alpha=0.8
        )
        plt.title(f"SupCon: {task_id}")
        plt.legend(bbox_to_anchor=(1.05, 1), loc=2)
        plt.tight_layout()
        
        save_path = OUTPUT_DIR / f"{task_id}_pca.png"
        plt.savefig(save_path)
        plt.close() # 关闭图片防止内存溢出
        print(f"Saved plot to {save_path}")

# --- 5. 保存最终数据供 RSA 使用 ---
import pickle
output_pkl = OUTPUT_DIR / "all_supcon_embeddings.pkl"
with open(output_pkl, 'wb') as f:
    pickle.dump(final_embeddings, f)

print(f"\nAll done! Embeddings saved to {output_pkl}")
print(f"Keys available: {list(final_embeddings.keys())}")


Loading Data: Belief_Period_H5 / Left_PFC.h5
Training Belief_Period_H5_Left_PFC...
Saved plot to results_supcon/Belief_Period_H5_Left_PFC_pca.png

Loading Data: Belief_Period_H5 / Left_TPJ.h5
Training Belief_Period_H5_Left_TPJ...
Saved plot to results_supcon/Belief_Period_H5_Left_TPJ_pca.png

Loading Data: Belief_Period_H5 / Right_PFC.h5
Training Belief_Period_H5_Right_PFC...
Saved plot to results_supcon/Belief_Period_H5_Right_PFC_pca.png

Loading Data: Belief_Period_H5 / Right_TPJ.h5
Training Belief_Period_H5_Right_TPJ...
Saved plot to results_supcon/Belief_Period_H5_Right_TPJ_pca.png

Loading Data: Perspective_Period_H5 / Left_PFC.h5
Training Perspective_Period_H5_Left_PFC...
Saved plot to results_supcon/Perspective_Period_H5_Left_PFC_pca.png

Loading Data: Perspective_Period_H5 / Left_TPJ.h5
Training Perspective_Period_H5_Left_TPJ...
Saved plot to results_supcon/Perspective_Period_H5_Left_TPJ_pca.png

Loading Data: Perspective_Period_H5 / Right_PFC.h5
Training Perspective_Period_H5